In [ ]:
# [Colab 셀 0] 한글 폰트 설치 및 한글 마이크로LED 공정 매뉴얼 PDF 자동 생성
!sudo apt-get install -y -qq fonts-nanum > /dev/null
!pip install -q reportlab

import os
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(font_path):
    pdfmetrics.registerFont(TTFont("NanumGothic", font_path))
    font_name = "NanumGothic"
    print("나눔고딕 한글 폰트 등록 완료")
else:
    font_name = "Helvetica"
    print("나눔고딕을 찾을 수 없어 Helvetica를 사용합니다.")

def create_korean_manual_pdf(filename="micro_led_manual.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    manual_pages = [
        [
            "[제 1장: 마이크로LED 공정 개요 및 표준 품질 기준]",
            "",
            "1. 레이저 리프트오프 (LLO: Laser Lift-Off) 공정 기준:",
            "  - UV 레이저의 에너지 밀도는 850 +/- 20 mJ/cm2 범위 내에서 제어되어야 한다.",
            "  - 에너지가 830 mJ/cm2 미만일 경우 GaN 박막 분리 불량이 발생할 수 있다.",
            "  - 에너지가 870 mJ/cm2 초과 시 LED 칩에 열 손상 및 미세 크랙이 발생한다.",
            "",
            "2. GaN 박막 두께 검사 기준:",
            "  - 공정 후 허용 두께 범위는 4.5 +/- 0.2 마이크로미터이다.",
            "  - 웨이퍼당 9개 지점에서 엘립소메트리 계측을 필수적으로 수행해야 한다."
        ],
        [
            "[제 2장: MOCVD 장비 공정 이상 알람 코드 조치 매뉴얼]",
            "",
            "1. 알람 코드: ERR-404 (챔버A 압력 및 온도 상한 이탈 알람)",
            "  - 발생 조건: 챔버A 온도가 820도(C)를 초과하거나 압력이 53 Torr를 넘을 때 발생.",
            "  - 물리적 메커니즘: 과도한 온도는 박막의 결정성 불량을 유발하며, NH3 유량이",
            "    불균일해지면서 기판 상단에 크랙(Crack) 및 흰색 백화 결함이 형성됨.",
            "  - 즉각 긴급 조치 SOP:",
            "    (1) RF 가열 유도 장치의 출력을 현재 수치에서 즉시 5% 하향 조정한다.",
            "    (2) NH3 유량 조절 MFC 밸브 B-203의 개폐도를 정확히 2.0 slm으로 고정한다.",
            "    (3) 챔버 압력이 50 Torr로 안정화될 때까지 웨이퍼 투입 라인을 대기 상태로 전환한다.",
            "",
            "2. 알람 코드: ERR-502 (TMGa 유량 불안정 알람)",
            "  - 발생 조건: 갈륨 소스(TMGa) 버블러 유량이 140 umol/min 이하로 하락 시 발생.",
            "  - 조치 방법: 운반 가스(H2) 압력을 재검사하고 소스 칠러 온도를 -5도(C)로 리셋한다."
        ],
        [
            "[제 3장: 설비 예방 보전(PM) 주기 및 유지보수 표준]",
            "",
            "1. 챔버A 정기 클리닝 주기 및 절차:",
            "  - 주기: 매 공정 가동 100시간마다 필수적으로 진행해야 한다.",
            "  - 절차: NF3 플라즈마 퍼지를 30분간 수행한 후, 서스셉터 표면 이물질을 검사한다.",
            "",
            "2. 서스셉터 교체 표준 기준:",
            "  - 표면 거칠기(Ra)가 0.5 마이크로미터를 초과할 경우 즉시 교체해야 한다.",
            "  - 교체 후에는 반드시 초기화 베이킹(Baking) 공정을 2시간 진행해야 한다."
        ]
    ]
    for page_lines in manual_pages:
        textobject = c.beginText(50, 740)
        textobject.setFont(font_name, 11)
        textobject.setLeading(18)
        for line in page_lines:
            textobject.textLine(line)
        c.drawText(textobject)
        c.showPage()
    c.save()
    print(f"한글 마이크로LED 공정 매뉴얼 PDF 생성 완료: '{filename}'")

create_korean_manual_pdf("micro_led_manual.pdf")


debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.0 MB/s eta 0:00:00
나눔고딕 한글 폰트 등록 완료
한글 마이크로LED 공정 매뉴얼 PDF 생성 완료: 'micro_led_manual.pdf'


In [ ]:
# [Colab 셀 1] 실습 ① — 문서 로드와 청킹
!pip install -q transformers torch scikit-learn faiss-cpu langchain langchain-community langchain-core langchain-text-splitters pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("micro_led_manual.pdf")
docs = loader.load()
print(f"총 {len(docs)}페이지 로드 완료")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)
chunks = splitter.split_documents(docs)

avg_len = int(sum(len(c.page_content) for c in chunks) / len(chunks))
print(f"총 {len(chunks)}개 청크, 평균 {avg_len}자")

print("\n--- [첫 번째 청크 샘플 확인] ---")
print("Page Content:\n", chunks[0].page_content[:200], "...")
print("Metadata:", chunks[0].metadata)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_700/2241639135.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


총 3페이지 로드 완료
총 3개 청크, 평균 383자

--- [첫 번째 청크 샘플 확인] ---
Page Content:
 [제 1장: 마이크로LED 공정 개요 및 표준 품질 기준]
1. 레이저 리프트오프 (LLO: Laser Lift-Off) 공정 기준:
  - UV 레이저의 에너지 밀도는 850 +/- 20 mJ/cm2 범위 내에서 제어되어야 한다.
  - 에너지가 830 mJ/cm2 미만일 경우 GaN 박막 분리 불량이 발생할 수 있다.
  - 에너지가 870 mJ/cm2 ...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-08-13T06:16:25+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-08-13T06:16:25+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'micro_led_manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


In [ ]:
# [Colab 셀 2] 실습 ② — FAISS Vector DB 구축
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import torch, os

print("'BAAI/bge-m3' 임베딩 모델 다운로드 및 로드 중...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("청크 임베딩 변환 및 FAISS 인덱스 생성 중...")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_microled_index")
print(f"Vector DB 저장 완료: {len(chunks)}개 청크 인덱싱")


'BAAI/bge-m3' 임베딩 모델 다운로드 및 로드 중...


/tmp/ipykernel_700/3271291957.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

청크 임베딩 변환 및 FAISS 인덱스 생성 중...
Vector DB 저장 완료: 3개 청크 인덱싱


In [ ]:
# [Colab 셀 3] FAISS 인덱스 저장 검증 및 유사도 검색 테스트
if os.path.exists("faiss_microled_index"):
    print("'faiss_microled_index' 폴더 저장 완료:", os.listdir("faiss_microled_index"))

query = "ERR-404 알람 발생 시 조치 방법은?"
search_results = vectorstore.similarity_search(query, k=1)

print(f"\n검색 질문: '{query}'")
print(search_results[0].page_content)
print(f"출처 메타데이터: {search_results[0].metadata}")


'faiss_microled_index' 폴더 저장 완료: ['index.faiss', 'index.pkl']

검색 질문: 'ERR-404 알람 발생 시 조치 방법은?'
[제 2장: MOCVD 장비 공정 이상 알람 코드 조치 매뉴얼]
1. 알람 코드: ERR-404 (챔버A 압력 및 온도 상한 이탈 알람)
  - 발생 조건: 챔버A 온도가 820도(C)를 초과하거나 압력이 53 Torr를 넘을 때 발생.
  - 물리적 메커니즘: 과도한 온도는 박막의 결정성 불량을 유발하며, NH3 유량이
    불균일해지면서 기판 상단에 크랙(Crack) 및 흰색 백화 결함이 형성됨.
  - 즉각 긴급 조치 SOP:
    (1) RF 가열 유도 장치의 출력을 현재 수치에서 즉시 5% 하향 조정한다.
    (2) NH3 유량 조절 MFC 밸브 B-203의 개폐도를 정확히 2.0 slm으로 고정한다.
    (3) 챔버 압력이 50 Torr로 안정화될 때까지 웨이퍼 투입 라인을 대기 상태로 전환한다.
2. 알람 코드: ERR-502 (TMGa 유량 불안정 알람)
  - 발생 조건: 갈륨 소스(TMGa) 버블러 유량이 140 umol/min 이하로 하락 시 발생.
  - 조치 방법: 운반 가스(H2) 압력을 재검사하고 소스 칠러 온도를 -5도(C)로 리셋한다.
출처 메타데이터: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-08-13T06:16:25+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-08-13T06:16:25+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'micro_led_manual.pdf', 'total_pages': 3, 'page': 1, 'page_la

In [ ]:
# [Colab 셀 4] ★ 수정본 — 최신 LCEL 방식의 단발성 RAG
#
# 핵심 수정:
#   기존: from langchain.chains import RetrievalQA  -> 삭제
#   현재: langchain_core.runnables + PromptTemplate + Retriever + LLM을 직접 연결
#
# 따라서 'ModuleNotFoundError: No module named langchain.chains'
# 오류가 발생하지 않습니다.

!pip install -q langchain-google-genai

from typing import Any, List, Optional
from langchain_core.language_models.llms import LLM
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

# -------------------------------------------------------------------------
# 실습용 시뮬레이션 LLM
# -------------------------------------------------------------------------
class PracticeSimulatedLLM(LLM):
    """API 키 없이도 RAG 파이프라인을 테스트할 수 있는 실습용 LLM."""

    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs: Any) -> str:
        if "ERR-404" in prompt:
            return ("ERR-404 알람 조치 방법은 1) RF 가열 유도 장치 출력을 즉시 5% 하향하고, "
                    "2) NH3 유량 조절 MFC 밸브 B-203을 2.0 slm으로 고정하며, "
                    "3) 압력이 50 Torr로 안정화될 때까지 웨이퍼 투입 라인을 대기 상태로 전환하는 것입니다.")
        elif "에너지 밀도" in prompt or "LLO" in prompt:
            return ("레이저 리프트오프(LLO) 공정의 UV 레이저 에너지 밀도 기준은 "
                    "850 +/- 20 mJ/cm2 범위 내에서 제어해야 합니다. "
                    "830 미만은 박막 분리 불량, 870 초과는 칩 열 손상을 유발합니다.")
        elif "두께" in prompt or "엘립소메트리" in prompt:
            return ("GaN 박막의 공정 후 허용 두께 범위는 4.5 +/- 0.2 마이크로미터이며, "
                    "웨이퍼당 9개 지점에서 엘립소메트리 계측을 필수 수행해야 합니다.")
        elif "클리닝" in prompt or "주기" in prompt:
            return ("챔버A 클리닝은 매 공정 가동 100시간마다 필수적으로 진행해야 하며, "
                    "NF3 플라즈마 퍼지를 30분간 수행한 후 서스셉터 표면을 검사합니다.")
        else:
            return "공정 매뉴얼 내용을 기반으로 답변합니다: " + prompt[-150:]

    @property
    def _llm_type(self) -> str:
        return "practice_simulated_llm"

# -------------------------------------------------------------------------
# LLM 선택
# 기본값: API 키가 없어도 실행되는 시뮬레이션 LLM
# 실제 Gemini를 사용하려면 USE_GEMINI = True로 변경하세요.
# -------------------------------------------------------------------------
USE_GEMINI = True

if USE_GEMINI:
    import os
    from getpass import getpass
    from langchain_google_genai import ChatGoogleGenerativeAI

    # if not os.environ.get("GOOGLE_API_KEY"):
    #     os.environ["GOOGLE_API_KEY"] = getpass("GOOGLE_API_KEY를 입력하세요: ")
    os.environ["GOOGLE_API_KEY"] = "-"

    # 필요에 따라 현재 사용 가능한 Gemini 모델명으로 변경
    GEMINI_MODEL = "gemini-3.6-flash"
    llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)
    print(f"실제 Gemini LLM 로드 완료: {GEMINI_MODEL}")
else:
    llm = PracticeSimulatedLLM()
    print("시뮬레이션 LLM 로드 완료")

# -------------------------------------------------------------------------
# Retriever + 프롬프트
# -------------------------------------------------------------------------
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)

prompt = PromptTemplate.from_template(
    """당신은 마이크로LED 공정 전문 엔지니어입니다.
반드시 [공정 매뉴얼 내용]만 바탕으로 답변하세요.
없는 내용은 '해당 내용이 없습니다'라고 답변하세요.

[공정 매뉴얼 내용]
{context}

[질문]
{question}

[답변]
"""
)

def format_docs(docs):
    return "\n\n".join(
        f"[문서 {i+1}]\n{doc.page_content}" for i, doc in enumerate(docs)
    )

def answer_rag(question, retriever=retriever, llm=llm, prompt=prompt):
    """RetrievalQA를 사용하지 않는 간단하고 투명한 LCEL/함수형 RAG."""
    retrieved_docs = retriever.invoke(question)
    context = format_docs(retrieved_docs)
    final_prompt = prompt.invoke({"context": context, "question": question})
    response = llm.invoke(final_prompt)

    # ChatGoogleGenerativeAI는 AIMessage, 일반 LLM은 문자열을 반환할 수 있음
    answer = response.content if hasattr(response, "content") else str(response)

    return {
        "result": answer,
        "source_documents": retrieved_docs
    }

query_1 = "ERR-404 알람 발생 시 조치 방법은?"
result_1 = answer_rag(query_1)

print("[단발성 질문]:", query_1)
print("[RAG 답변]:", result_1["result"])
print("[참조 문서]:")
for idx, doc in enumerate(result_1["source_documents"], 1):
    print(f"  ({idx}) {doc.metadata.get('source', '매뉴얼')} / page={doc.metadata.get('page', 'N/A')}")
    print("      ", doc.page_content[:80].replace("\n", " "), "...")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 26.6 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/chat_models.py:431: UserWarning: Unrecognized keys in model profile: ['reasoning_effort_default', 'reasoning_effort_levels']. This may indicate a version mismatch between langchain-core and your provider package. Consider upgrading langchain-core.
  _warn_unknown_profile_keys(self.profile)


실제 Gemini LLM 로드 완료: gemini-3.6-flash


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[단발성 질문]: ERR-404 알람 발생 시 조치 방법은?
[RAG 답변]: [{'type': 'text', 'text': 'ERR-404 알람 발생 시 즉각 긴급 조치 SOP는 다음과 같습니다.\n\n1. RF 가열 유도 장치의 출력을 현재 수치에서 즉시 5% 하향 조정합니다.\n2. NH3 유량 조절 MFC 밸브 B-203의 개폐도를 정확히 2.0 slm으로 고정합니다.\n3. 챔버 압력이 50 Torr로 안정화될 때까지 웨이퍼 투입 라인을 대기 상태로 전환합니다.', 'extras': {'signature': 'EoIPCv8OARFNMg8lTtUxgNCJPH07/lmvfH9Sb59PGcB6TI7Tmz/XOV4sA4mWEJkefloDZOEQI1XhFJ3DxvMKRFp80nfPzVwpsjbMyoS3F+lUoO/zJD9/acDuRlVuQKfx6vxy76ALIZEFaZKr3sz5AAjq9EourR5NBlU3UWowaMK3D/WcgeZSjcCdDILceSsaBTT9Vwl8sfpZrl6EqjH/9NsXuhdQi+opvYPwte4MmYav8rKaE9yCoDA2AeIIDwLg6WlaJMNV+uNaVT2xKykbZh+/DOFj+Q0iqrTh7Y3B68uCFgLkjcxDRpfx+SHdzPmx7DvNwoBp6KqKRHDeyelatOENoEE5n6M/41ykWsV/VODkq+Jd2C0KY0FS/XJqu+xdVnBehjXcYEyZzW8ZwFYU4jLBpGU1hCUvcqaV/hAcTTcg9gYaC0wVoTMVTqSqgIRh+nzBN/fKdDGmzLhAvE32KE/PDTw+HbK/ZBYQuplhDBamqlc3D7zmKBO2STxiL+30O1tnji3iS3kwrNclAKhpFeOA62lfDRywXSeDU+zBt7bs3M2Y+RbfeW7iSIfgRv6rXysXaVWKrs4vQ//xHCEJyGCU7P+g6JIfSAeN2Yn7lnuuHYxGp9rCJylbdgZo+5v3SJfd5q1J+ap3tMTYxqyQXxWUocSQabmdKkJMNxcKBsCH1NlMF9q7vlecc3tx8hA39/0

In [ ]:
# [Colab 셀 5] 실습 — 메타데이터 필터링
print("=== [메타데이터 필터링 실습] ===")

for chunk in chunks:
    if "ERR" in chunk.page_content or "알람" in chunk.page_content:
        chunk.metadata.update({"equipment": "MOCVD", "doc_type": "maintenance_manual"})
    else:
        chunk.metadata.update({"equipment": "General", "doc_type": "standard_spec"})

vectorstore_meta = FAISS.from_documents(chunks, embeddings)

retriever_filtered = vectorstore_meta.as_retriever(
    search_kwargs={
        "k": 2,
        "filter": {"equipment": "MOCVD"}
    }
)

query_filter = "ERR-404 알람 조치 방법은?"
filtered_docs = retriever_filtered.invoke(query_filter)

print(f"질문: {query_filter}")
print(f"MOCVD 필터링 검색 결과: {len(filtered_docs)}개")
for idx, doc in enumerate(filtered_docs, 1):
    print(f"  - [{idx}위] 메타데이터: {doc.metadata}")
    print(f"  - 내용 요약: {doc.page_content[:80].replace(chr(10), ' ')}...")


=== [메타데이터 필터링 실습] ===
질문: ERR-404 알람 조치 방법은?
MOCVD 필터링 검색 결과: 1개
  - [1위] 메타데이터: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-08-13T06:16:25+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-08-13T06:16:25+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'micro_led_manual.pdf', 'total_pages': 3, 'page': 1, 'page_label': '2', 'equipment': 'MOCVD', 'doc_type': 'maintenance_manual'}
  - 내용 요약: [제 2장: MOCVD 장비 공정 이상 알람 코드 조치 매뉴얼] 1. 알람 코드: ERR-404 (챔버A 압력 및 온도 상한 이탈 알람)   -...


In [ ]:
# [Colab 셀 6] ★ 수정본 — Multi-turn RAG
#
# 기존:
#   from langchain.memory import ConversationBufferMemory
#   from langchain.chains import ConversationalRetrievalChain
#
# 위 두 모듈은 최신 LangChain 환경에서 버전 차이/이동으로 오류가 발생할 수 있으므로
# 이 실습에서는 대화 이력을 파이썬 리스트로 명시적으로 관리합니다.
# 교육 목적에서도 '대화 이력 -> 검색 질의 -> 컨텍스트 -> 답변' 흐름을 쉽게 볼 수 있습니다.

print("=== [대화 기록 유지 Multi-turn RAG 실습] ===")

chat_history = []

multi_turn_prompt = PromptTemplate.from_template(
    """당신은 마이크로LED 공정 전문 엔지니어입니다.
반드시 [공정 매뉴얼 내용]과 [대화 기록]을 근거로 답변하세요.
매뉴얼에 없는 내용은 '해당 내용이 없습니다'라고 답변하세요.

[대화 기록]
{history}

[공정 매뉴얼 내용]
{context}

[현재 질문]
{question}

[답변]
"""
)

def history_to_text(history):
    if not history:
        return "(이전 대화 없음)"
    return "\n".join(
        f"{role}: {text}" for role, text in history[-6:]
    )

def answer_multiturn(question):
    # 대명사/생략 질문의 검색 정확도를 높이기 위해 최근 대화를 검색 질의에 함께 사용
    search_query = history_to_text(chat_history) + "\n현재 질문: " + question
    retrieved_docs = retriever_filtered.invoke(search_query)
    context = format_docs(retrieved_docs)

    final_prompt = multi_turn_prompt.invoke({
        "history": history_to_text(chat_history),
        "context": context,
        "question": question
    })

    response = llm.invoke(final_prompt)
    answer = response.content if hasattr(response, "content") else str(response)

    chat_history.append(("사용자", question))
    chat_history.append(("AI", answer))

    return {"answer": answer, "source_documents": retrieved_docs}

q1 = "ERR-404 알람의 원인이 뭐야?"
print(f"\n엔지니어 (질문 1): {q1}")
r1 = answer_multiturn(q1)
# print(f"AI 답변 1: {r1['answer'].strip()}")
print(f"AI 답변 1: {r1['answer']}")

q2 = "그럼 어떻게 조치해야 해?"
print(f"\n엔지니어 (질문 2): {q2}")
r2 = answer_multiturn(q2)
# print(f"AI 답변 2: {r2['answer'].strip()}")
print(f"AI 답변 2: {r2['answer']}")

print("\n[대화 이력]")
for role, text in chat_history:
    print(f"{role}: {text}")


=== [대화 기록 유지 Multi-turn RAG 실습] ===

엔지니어 (질문 1): ERR-404 알람의 원인이 뭐야?


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AI 답변 1: [{'type': 'text', 'text': 'ERR-404 알람은 **챔버A 온도가 820도(C)를 초과하거나 압력이 53 Torr를 넘을 때** 발생합니다. \n\n이때 과도한 온도로 인해 박막의 결정성 불량이 유발되며, NH3 유량이 불균일해지면서 기판 상단에 크랙(Crack) 및 흰색 백화 결함이 형성됩니다.', 'extras': {'signature': 'ErwbCrkbARFNMg+2dJo6qmA3A0URLoYDiKSSgTpEQqo/djnIXcqE22gMZkB5Yn/EFMBeWHv7FwKw4vvsk1n0+PqEme+mg/M21Hdt197pj31I2igXxAvtmW3fyXkIxb1Uyy9y9saXSXF9AKPQ4xO0HWNyhZ1BTi8C0Tr1xVAJHmXOlFYBblcT0FYl1+Obn6on/l3UjI5pWTytX/NIh/4EvW//nl05ovliu9iIjqorSfvOmQ6KbOtsQDMCxO6jm9OQEcQZCGFaBmqyaA03ZBqdKccdtrdvsR68fKkzJUkMTdF6lE0sJOAorvmWHn5zBI0Wdt0ezBOUhZcAb81AqVYIk5EPRK1vQnj6Mzxf88hTKEAXpO6GcaLpz9WUfXB+jZAr7Kq8MTHHa61EoulbtQzJBeVnPkmega3o6LPxeWtQHzQ08BRFR/Fuz4IxUhuLXF7tyWjFd+jcSkupH5x2b98LQXmivBbLv/Cca6JFIC4sSBFrXuWdqjz+O2BV7tLGy2P9TUhizQrsxM+fAxztHgP5YUbeR7cRZoOQSJq7NAY2kankdzZC1oUGATV7qsicVnNKCyTkzh7P9nqGu7sKm751hRZM+yHW6Q4b65UCt04o8ibAagaS81cEnUZysbkK4rbx+pfhZdJ4U4TSNTDMYhv+Ti0ATeA1xhAq26kDVgtCLXO37Vp1X9mwm0k0yPITAmLwK4qEQi7rpJ4okgJhBSLi8DLn/PkCNFnV8kpKKs8IeqTFdrIRRYVo+VeMRkz7rN7i8hjL8bqxZS5L3081k

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


ChatGoogleGenerativeAIError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 40.964799388s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
# [Colab 셀 7] ★ 수정본 — 전체 RAG 파이프라인 종합 성능 검증
print("========================================================")
print("   [마이크로LED 공정 RAG 파이프라인 종합 성능 검증 리포트]")
print("========================================================\n")

test_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

test_questions = [
    "1. 레이저 리프트오프 공정에서 에너지 밀도 기준은?",
    "2. ERR-404 알람 발생 시 즉각 조치 방법은?",
    "3. GaN 박막 두께 측정 방법과 허용 오차는?",
    "4. 챔버 클리닝 주기와 절차는?",
    "5. 불량률이 갑자기 증가했을 때 체크리스트는?"
]

for q in test_questions:
    print(f"질문: {q}")

    result = answer_rag(q, retriever=test_retriever)
    answer = result["result"]
    sources = result["source_documents"]

    # print(f"답변: {answer.strip()}")
    print(f"답변: {answer}")

    if sources:
        top_doc = sources[0]
        page_num = top_doc.metadata.get("page", "N/A")
        print(
            f"근거 출처: 매뉴얼 {page_num}페이지 "
            f"(내용: {top_doc.page_content[:50].replace(chr(10), ' ')}...)"
        )
    else:
        print("근거 출처: 검색된 문서 없음")

    print("-" * 65)

print("\n[RAG 시스템 평가 포인트]")
print("1. 충실도(Faithfulness): 답변이 검색된 근거와 일치하는가?")
print("2. 환각 방지(Hallucination): 문서에 없는 질문에 임의로 답하지 않는가?")
print("3. 검색 품질(Retrieval): 질문과 관련된 청크가 상위에 검색되는가?")
print("4. 출처 추적(Source): 답변 근거 문서의 페이지/메타데이터를 확인할 수 있는가?")
